# Блокнот 01: Профілювання якості даних та первинне очищення

**Фаза 1 нашого проєкту (Data Ingestion & Quality Gate 1)**

У цьому блокноті реалізовано:
1. Завантаження сирого датасету `data/raw/official_data_uk.csv`.
2. Фільтрація за рівнем `level == 'oblast'` для консистентності 2022–2026 років.
3. Конвертація часових поясів (UTC -> `Europe/Kyiv`).
4. Виявлення та фільтрація аномалій (мікро-глітчі < 60с, записи з ТОТ).
5. Хронологічний 3-way спліт (Train / Validation / Test).
6. Верифікація за критеріями **Quality Gate 1**.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

parquet_path = Path("../data/processed/alerts_clean_oblast.parquet")
df = pd.read_parquet(parquet_path)
print(f"Завантажено очищених рядків: {len(df):,}")
df.head()

## 1. Загальна інформація про датасет

In [ ]:
print(f"Діапазон спостережень: {df['started_at_kyiv'].min()} — {df['started_at_kyiv'].max()}")
print(f"Кількість унікальних областей: {df['oblast'].nunique()}")
print("\nРозподіл за вибірками хронологічного спліту:")
print(df['split_set'].value_counts())
df.info()

## 2. Аналіз тривалості тривог (хвилини)

In [ ]:
regular_df = df[~df["is_occupied_extreme"] & df["is_finished"]]

print("Статистика тривалості (хвилини):")
print(regular_df["duration_minutes"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]))

## 3. Топ областей за кількістю оголошених тривог

In [ ]:
oblast_counts = regular_df["oblast"].value_counts()
print("Топ-5 найбільш активних областей:")
print(oblast_counts.head(5))

print("\nТоп-5 найбезпечніших областей (глибокий тил):")
print(oblast_counts.tail(5))

## 4. Перевірка Quality Gate 1

In [ ]:
assert len(df) > 120000, "Gate 1 Fail: Row count too low!"
assert (df[df["is_finished"]]["duration_seconds"] < 0).sum() == 0, "Gate 1 Fail: Negative durations!"
assert df["started_at_kyiv"].notna().all(), "Gate 1 Fail: Missing timezone conversions!"
print("🎉 QUALITY GATE 1 УСПІШНО ПРОЙДЕНО!")